                                    Section 2 Supervised Learning


-----------------------------------

In [15]:
import string
import base64
import pandas as pd
import re
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression

dataSet = pd.read_parquet(r"C:\Users\A.RUSSO\PycharmProjects\JupyterProject\data\ssh_attacks.parquet")

def decode_base64_payload(payload):
    base64_pattern = r'([A-Za-z0-9+/]{20,}={0,2})'
    if not "base64" in payload:
        return payload
    matches = re.findall(base64_pattern, payload)
    decoded_strings = []
    for match in matches:
        try:
            decoded = base64.b64decode(match).decode('utf-8', errors='ignore')
            # Filter out non-printable characters
            if all(c in string.printable for c in decoded):
                decoded_strings.append(decoded)
        except Exception as e:
            continue
    # return the original payload but with the decoded b64
    result = payload
    for match, decoded in zip(matches, decoded_strings):
        result = result.replace(match, decoded, 1)
    return result

dataSet['decoded_payload'] = dataSet['full_session'].apply(decode_base64_payload)

In [21]:

newDataSet = dataSet.drop(columns=['full_session','session_id'])
newDataSet["first_timestamp"] = pd.to_datetime(dataSet["first_timestamp"])
newDataSet["day_of_the_week"] = newDataSet["first_timestamp"].dt.dayofweek
newDataSet["month"] = newDataSet["first_timestamp"].dt.month



y = newDataSet['Set_Fingerprint']

newDataSet.drop(columns=['Set_Fingerprint'], inplace=True)


X_train, X_test, y_train, y_test = train_test_split(
    newDataSet,
    y,
    train_size=0.7,
    random_state=42,
)


preprocessor = ColumnTransformer(
    transformers=[
        (
            "full_session_vectorized",
            TfidfVectorizer(
                lowercase=True,
                stop_words="english"
            ),
            "decoded_payload"
        ),
        (
            "time",
            StandardScaler(),
            ["day_of_the_week", "month"]
        )
    ]
)

X_train_preprocessed = preprocessor.fit_transform(X_train)




[-1.43987297 -0.49840799]
